# 节点 2：状态、周期与直接勾选

这一节点把用户设定的周期变成每天都会自动变化的状态，并让首页勾选直接形成记录。

## 1. 本节点目标

系统读取运行当天日期，计算距离上次清洗或晾晒有几天，再与用户自己的周期比较。用户勾选“今天已清洗/晾晒”后，SQLite 会记录今天并立即重算状态。

## 2. 完成结果与验收

- 清洗和晾晒分别计算天数、比例、表情、文案与是否到期。
- 从未记录、0.6、1.0、1.5 边界和未来日期都有测试。
- 首页可以直接勾选今天完成，重复提交不会重复写入。
- 写活动历史与更新最近日期处于同一事务。
- 删除物品会级联删除其活动记录。

## 3. 本节点文件结构

```text
src/smart_laundry/status_rules.py   纯日期与周期规则
src/smart_laundry/database.py       activity_records 表与外键
src/smart_laundry/repositories.py   写活动记录并更新最近日期
app.py                              状态卡片与直接勾选
tests/test_status_rules.py          固定日期和边界测试
tests/test_activity_records.py      事务、幂等和级联测试
```

## 4. 关键代码解释

源文件：`src/smart_laundry/status_rules.py`。`raw_days = (current_date - performed_date).days` 得到已经过去的自然日；`ratio = raw_days / interval_days` 把不同周期转换成可比较比例。比例小于 0.6 是良好，达到 1.0 表示到期，达到 1.5 表示超期较久。

源文件：`src/smart_laundry/repositories.py`。`record_activity()` 先插入活动记录，再更新物品最近日期。二者使用同一数据库连接：成功一起提交，失败一起回滚。唯一约束避免同一物品、同一动作、同一时间重复记录。

In [ ]:
from datetime import date

today = date(2026, 8, 29)
last_done = date(2026, 8, 19)
interval_days = 10
days_since = (today - last_done).days
ratio = days_since / interval_days
print(days_since, ratio)  # 10 天，比例 1.0，正好到期

## 5. 数据流

```mermaid
flowchart LR
 A[页面勾选今天已清洗] --> B[record_activity]
 B --> C[(activity_records 新记录)]
 B --> D[items.last_washed_at 更新]
 D --> E[status_rules 读取今天与周期]
 E --> F[天数 状态 表情 原因]
 F --> G[首页卡片立即刷新]
```

## 6. 关键概念

- **自然日差**：两个日期相减得到的天数。
- **比例**：已经过天数除以用户周期。
- **纯函数**：只根据输入返回结果，便于固定今天测试。
- **活动记录**：每次清洗或晾晒的历史事实。
- **幂等写入**：重复点击不会制造同一天的重复记录。
- **外键级联**：删除物品时自动清理它的活动历史。

## 7. 为什么这样设计

状态不依赖大模型，因为日期和比例需要稳定、可解释、可测试。`today` 可以作为参数传入，让测试不受真实日期变化影响；页面不传时才使用设备当天日期。勾选代表“记录今天完成”，而不是永久开关，所以新的一天会出现新的勾选项。

## 8. 常见错误与排查

1. **天数不对**：检查设备日期、时区和数据库中的 ISO 日期。
2. **勾选后没有变化**：查看页面成功提示，并确认数据库未锁定。
3. **重复产生历史**：检查唯一约束是否存在，以及 `performed_at` 格式是否一致。
4. **出现负天数**：未来记录会显示“记录日期在未来”，不会显示负数。
5. **清洗覆盖晾晒**：两种动作必须更新不同字段。

## 9. 面试可能追问

**问：为什么状态计算不交给 LLM？** 答：这是确定性业务事实，规则更稳定、便宜且可测试。追问可能是规则如何配置化。

**问：如何防止 Streamlit 重跑重复写入？** 答：页面先判断今天是否已记录，数据库再用唯一约束兜底。

**问：如何保证活动记录和最近日期一致？** 答：两个写操作在一个事务中提交或回滚。

## 10. 必须掌握的最少知识

需要理解日期可以相减、比例用于比较不同周期、清洗与晾晒是两条独立记录、事务保证数据不写一半。暂时不需要掌握复杂时区库。

## 11. 可自测小题

1. 上次清洗 10 天前、周期 10 天，比例是多少？
2. 为什么测试要传入固定 `today`？
3. 同一天重复勾选为什么不会重复写入？
4. 为什么活动历史和最近日期要放进同一事务？

<details><summary>参考答案</summary>

1. 1.0，正好到期。2. 让结果可重复。3. 页面判断加数据库唯一约束。4. 防止一个成功、另一个失败。

</details>

## 12. 动手小练习

1. 修改上方示例的周期为 20 天，观察比例。
2. 在页面把某物品周期设为 1 天，观察下一天状态变化。
3. 新增测试用例验证 0.59 与 0.60 附近的边界。

## 13. 本节点术语表

| 术语 | 简单解释 |
|---|---|
| ratio | 已过天数与周期的比值 |
| boundary | 状态切换的临界值 |
| idempotent | 重复执行结果保持稳定 |
| activity record | 一次完成动作的历史 |
| cascade delete | 删除主体时同步删除从属数据 |
| rollback | 出错后撤销本次事务 |

## 14. 下一节点连接

天气节点会把今日晴雨、湿度和降雨概率加入页面。状态已经到期且天气晴朗时，首页会把物品表情切换为愤怒提醒。